1. Import library

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from typing import Dict, List, Optional
import os

2. Generate Synth-data

In [2]:
# ==============================================
# XGBOOST SYNTHESIZER WITH PREDICT_PROBA
# ==============================================
class XGBoostSynthesizer:
    """
    Incremental/autoregressive tabular synthesizer
    - Train on real dataframe
    - Sampling on partial synthetic dataframe
    - Column by column
    - Adjustable noise
    """

    def __init__(
        self,
        random_state: int = 42,
        add_noise: bool = False,
        noise_factor: float = 0.05,
        n_estimators: int = 300,
        max_depth: int = 6
    ):
        self.random_state = random_state
        self.add_noise = add_noise          # If True adds noise
        self.noise_factor = noise_factor    # Fraction of std for noise
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.label_encoders: Dict[str, LabelEncoder] = {}
        self.original_dtypes: Dict[str, str] = {}
        self.real_columns: List[str] = []
        self.df_real: Optional[pd.DataFrame] = None

        pd.options.mode.copy_on_write = True
        np.random.seed(self.random_state)

    # =====================================================
    # PUBLIC API
    # =====================================================
    def fit_sample(
        self,
        df: pd.DataFrame,
        sample_size: Optional[int] = None,
        column_order: Optional[List[str]] = None
    ) -> pd.DataFrame:

        if sample_size is None:
            sample_size = len(df)

        self.df_real = df.copy()
        self.real_columns = df.columns.tolist()
        self.original_dtypes = df.dtypes.to_dict()

        columns_data = self._generate_all_columns(
            df=df,
            sample_size=sample_size,
            column_order=column_order
        )

        df_synth = pd.DataFrame(columns_data)
        return self._restore_dtypes(df_synth)

    # =====================================================
    # CORE LOGIC
    # =====================================================
    def _generate_all_columns(
        self,
        df: pd.DataFrame,
        sample_size: int,
        column_order: Optional[List[str]]
    ) -> Dict[str, np.ndarray]:

        # Column order if not specified: first numeric then categorical
        if column_order is None:
            num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
            synthesis_order = num_cols + cat_cols
        else:
            synthesis_order = [c for c in column_order if c in df.columns]

        print(f"Incremental synthesis order: {synthesis_order}")

        columns_data: Dict[str, np.ndarray] = {}

        # ---------- FIRST COLUMN (PURE BOOTSTRAP)
        first_col = synthesis_order[0]
        columns_data[first_col] = df[first_col].sample(
            n=sample_size,
            replace=True,
            random_state=self.random_state
        ).values

        # ---------- SUBSEQUENT COLUMNS (INCREMENTAL)
        for col in synthesis_order[1:]:
            print(f"Synthesizing {col}")
            df_synth_partial = pd.DataFrame(columns_data)

            synth_col = self._synthesize_single_column(
                df_real=df,
                df_synth_partial=df_synth_partial,
                target_col=col
            )

            columns_data[col] = synth_col

        return columns_data

    def _synthesize_single_column(
        self,
        df_real: pd.DataFrame,
        df_synth_partial: pd.DataFrame,
        target_col: str
    ) -> np.ndarray:

        feature_cols = df_synth_partial.columns.tolist()

        # -----------------------------
        # TRAIN (REAL DATA)
        # -----------------------------
        X_train = self._prepare_features(df_real[feature_cols])
        y_train = df_real[target_col]

        # -----------------------------
        # SAMPLE (SYNTH DATA)
        # -----------------------------
        X_synth = self._prepare_features(df_synth_partial)
        X_synth = self._align_features(X_train, X_synth)

        is_categorical = (
            y_train.dtype in ["object", "category", "str"]
        )

        # =================================================
        # CATEGORICAL
        # =================================================
        if is_categorical:
            print("Categorical column detected")
            y_enc = self._encode_target(y_train, target_col)

            model = xgb.XGBClassifier(
                n_estimators=self.n_estimators,
                max_depth=self.max_depth,
                learning_rate=0.05,
                subsample=0.7,
                colsample_bytree=0.7,
                eval_metric="mlogloss",
                random_state=self.random_state
            )

            model.fit(X_train, y_enc)
            
            # Using predict_proba maintains natural data variability
            proba = model.predict_proba(X_synth)

            # Iterate over each row using probability array
            # Randomly choose with probability weighting
            sampled = np.array([
                np.random.choice(len(p), p=p)
                for p in proba
            ])

            return self._decode_target(sampled, target_col)

        # =================================================
        # NUMERIC
        # =================================================
        else:
            print("Numeric column detected")
            model = xgb.XGBRegressor(
                n_estimators=self.n_estimators,
                max_depth=self.max_depth,
                learning_rate=0.05,
                subsample=0.7,
                colsample_bytree=0.7,
                random_state=self.random_state
            )

            model.fit(X_train, y_train)

            preds = model.predict(X_synth)
            print(f"Predictions shape: {len(preds)}")
            if self.add_noise:
                noise_std = max(np.std(y_train) * self.noise_factor, 1e-6)
                noise = np.random.normal(0, noise_std, size=len(preds))
                preds = preds + noise

            return np.clip(preds, y_train.min(), y_train.max())

    # =====================================================
    # UTILITIES
    # =====================================================
    def _prepare_features(self, df: pd.DataFrame) -> pd.DataFrame:
        return pd.get_dummies(df, prefix_sep="_")

    def _align_features(
        self,
        X_train: pd.DataFrame,
        X_synth: pd.DataFrame
    ) -> pd.DataFrame:

        missing = set(X_train.columns) - set(X_synth.columns)
        for col in missing:
            X_synth[col] = 0

        return X_synth[X_train.columns]

    def _encode_target(self, y: pd.Series, col_name: str) -> np.ndarray:
        if col_name not in self.label_encoders:
            le = LabelEncoder()
            self.label_encoders[col_name] = le.fit(y)
        return self.label_encoders[col_name].transform(y)

    def _decode_target(self, y_enc: np.ndarray, col_name: str) -> np.ndarray:
        return self.label_encoders[col_name].inverse_transform(y_enc)

    def _restore_dtypes(self, df_synth: pd.DataFrame) -> pd.DataFrame:
        df_out = df_synth.copy()

        for col, dtype in self.original_dtypes.items():
            if col not in df_out.columns:
                continue

            try:
                dtype_str = str(dtype).lower()
                s = df_out[col]

                if "int" in dtype_str:
                    df_out[col] = s.fillna(0).round().astype("int64")
                elif "float" in dtype_str:
                    df_out[col] = s.astype("float64")
                elif "bool" in dtype_str:
                    df_out[col] = s.astype("boolean")
                elif "datetime" in dtype_str:
                    df_out[col] = pd.to_datetime(s, errors="coerce")
                else:
                    df_out[col] = s.astype("object")

            except Exception as e:
                print(f"Dtype error for {col}: {e}")

        return df_out.reindex(columns=self.real_columns)


# ==============================================
# COLUMN ORDER DEFINITIONS
# ==============================================

# Order 0: Default order
o0 = ['age', 'physical_activity', 'genetic_predisposition', 'municipality_residence', 
      'civil_status', 'gender', 'occupation', 'diagnosis']

# Order 1: Municipality first
o1 = ['municipality_residence', 'age', 'civil_status', 'gender', 'occupation', 
      'physical_activity', 'genetic_predisposition', 'diagnosis']

# Order 2: Demographic order
o2 = ['gender', 'age', 'physical_activity', 'civil_status', 'occupation', 
      'genetic_predisposition', 'municipality_residence', 'diagnosis']

# Order 3: Random-like order
o3 = ['municipality_residence', 'age', 'genetic_predisposition', 'physical_activity', 
      'civil_status', 'gender', 'occupation', 'diagnosis']

# Order 4: Reverse pyramid
o4 = ['occupation', 'gender', 'civil_status', 'physical_activity', 
      'genetic_predisposition', 'age', 'municipality_residence', 'diagnosis']

# Order 5: Increasing pyramid
o5 = ['physical_activity', 'genetic_predisposition', 'age', 'occupation', 
      'gender', 'municipality_residence', 'civil_status', 'diagnosis']

# Select order number from 0 to 5
selected_order = 0

order_list = [o0, o1, o2, o3, o4, o5]
optimal_order = order_list[selected_order]

print(optimal_order)
print(selected_order)


# ==============================================
# PATH CONFIGURATION FOR ONYXIA
# ==============================================

np.random.seed(42)

R = 'Y'  # Randomization setting

if R != "N":
    PR = 40  # Privacy risk parameter
else:
    PR = 0

synthesizer_type = "XGBoost"

# ONYXIA paths
input_dir = "/home/onyxia/work/data/Step1/Output"
output_dir = "/home/onyxia/work/data/Step2/Output"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

input_file_name = os.path.join(
    input_dir,
    f"real_data_datasetM10_TH20_R{R}_PR{PR}_4CAT_MISS_X2.csv"
)

output_file_name = os.path.join(
    output_dir,
    f"synthetic_data_datasetM10_TH20_R{R}_PR{PR}_4CAT_MISS_X2_{synthesizer_type}_O{selected_order}.csv"
)

print(input_file_name)
print(output_file_name)

# Load real data
df_real = pd.read_csv(input_file_name, sep=',')

# Drop unnecessary columns
columns_to_drop = ['id', 'score1', 'score2', 'bin1', 'bin2', 'raw_class', 
                   'age_norm', 'activity_norm', 'predisposition_norm',
                   'birth_year', 'birth_month', 'birth_day',
                   'municipality_birth', 'birth_dayofyear']
df_real = df_real.drop(columns=columns_to_drop)

# Convert to categorical columns as needed
numeric_cols = []  # Specify numeric columns if any
categorical_cols = ["physical_activity", "genetic_predisposition", "occupation", "age", "diagnosis"]

df_real[numeric_cols] = (
    df_real[numeric_cols]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .astype("int64")
)

# Transform to categorical
df_real[categorical_cols] = (
    df_real[categorical_cols]
    .astype(str)
    .replace(['nan', 'NaN', 'None', 'none', ''], 'missing')
    .fillna('missing')
    .apply(lambda x: x.str.strip())
)

print(df_real.dtypes)

# ==============================================
# SYNTHETIC DATA GENERATION
# ==============================================

# Initialize synthesizer
# Note: Set add_noise=True to add noise to numeric columns (5% of std)
synthesizer = XGBoostSynthesizer(random_state=42, add_noise=True)
df_synth = synthesizer.fit_sample(df_real, sample_size=10000, column_order=optimal_order)
df_synth.to_csv(output_file_name, sep=',', index=False)

print(df_synth.head())
print(df_real.head())
print(df_synth.dtypes)
print(df_real.dtypes)


# ==============================================
# QUALITY METRICS FUNCTIONS
# ==============================================

def comprehensive_comparison(df_real: pd.DataFrame, df_synth: pd.DataFrame):
    """
    Comprehensive scientific comparison between real and synthetic data
    Includes NMAE, TVD, KS, KL, Correlation, Privacy metrics
    """
    
    import numpy as np
    from scipy import stats
    
    print("\nSCIENTIFIC METRICS - LEGEND")
    print("NMAE  = Normalized Mean Absolute Error")
    print("TVD   = Total Variation Distance") 
    print("KS    = Kolmogorov-Smirnov (full distribution)")
    print("KL    = Kullback-Leibler Divergence")
    print("CORR  = Pearson Correlation preservation")
    print("SDFS  = Synthetic Data Fidelity Score")
    print("Privacy = Exact row match avoidance")
    print("-" * 70)
    
    print("SYNTHETIC DATA QUALITY REPORT")
    print("=" * 90)
    
    # Reset indices
    df_real_r = df_real.reset_index(drop=True)
    df_synth_r = df_synth.reset_index(drop=True)
    
    # Basic metrics
    print(f"\nDIMENSIONS")
    print(f"Real:     {df_real.shape}")
    print(f"Synthetic:{df_synth.shape}")
    print(f"Match:    {'YES' if df_real.shape == df_synth.shape else 'NO'}")
    
    print(f"\nDATA TYPES")
    dtype_match = df_real.dtypes.equals(df_synth.dtypes)
    print(f"Exact match: {'YES' if dtype_match else 'NO'}")
    
    # Numeric columns
    numeric_cols = [col for col in df_real.select_dtypes([np.number]).columns 
                   if df_real[col].nunique() > 1]
    
    print(f"\nNUMERIC COLUMNS ({len(numeric_cols)})")
    nmae_scores, ks_scores, kl_scores = {}, {}, {}
    
    for col in numeric_cols:
        r_data, s_data = df_real[col].dropna(), df_synth[col].dropna()
        
        # NMAE
        r_mean, s_mean = r_data.mean(), s_data.mean()
        r_std = r_data.std()
        nmae = abs(r_mean - s_mean) / max(r_std, 0.1)
        nmae_scores[col] = nmae
        
        # KS test
        ks_stat, _ = stats.ks_2samp(r_data, s_data)
        ks_scores[col] = ks_stat
        
        # KL Divergence
        try:
            r_hist, s_hist = np.histogram(r_data, bins=20, density=True)[0], np.histogram(s_data, bins=20, density=True)[0]
            kl = stats.entropy(r_hist + 1e-10, s_hist + 1e-10)
            kl_scores[col] = kl
        except:
            kl_scores[col] = 0
            
        status = "GOOD" if nmae < 0.1 and ks_stat < 0.1 else "MEDIUM" if nmae < 0.3 else "POOR"
        print(f"  {col:15}: NMAE:{nmae:.3f} KS:{ks_stat:.3f} KL:{kl_scores[col]:.2f} | {(1-nmae)*100:4.1f}% {status}")
    
    nmae_mean = np.mean(list(nmae_scores.values()))
    ks_mean = np.mean(list(ks_scores.values()))
    
    # Categorical columns - TVD
    print(f"\nCATEGORICAL COLUMNS - TVD")
    cat_cols = df_real.select_dtypes(exclude=[np.number]).columns
    tvd_scores, kl_cat_scores = {}, {}
    
    for col in cat_cols:
        r_vc = df_real[col].value_counts(normalize=True)
        s_vc = df_synth[col].value_counts(normalize=True)
        tvd = abs(r_vc.reindex(s_vc.index, fill_value=0) - s_vc).sum() / 2
        tvd_scores[col] = tvd
        
        # KL for categorical
        kl = stats.entropy(r_vc + 1e-10, s_vc.reindex(r_vc.index, fill_value=1e-10) + 1e-10)
        kl_cat_scores[col] = kl
        
        status = "GOOD" if tvd < 0.05 else "MEDIUM" if tvd < 0.15 else "POOR"
        print(f"  {col[:15]:15}: TVD:{tvd:.3f} KL:{kl:.2f} | {(1-tvd)*100:4.1f}% {status}")
    
    tvd_mean = np.mean(list(tvd_scores.values()))
    
    # Correlation preservation
    print(f"\nCORRELATION PRESERVATION")
    if len(numeric_cols) >= 2:
        corr_cols = numeric_cols[:4]
        r_corr_matrix = df_real[corr_cols].corr().values
        s_corr_matrix = df_synth[corr_cols].corr().values

        pairs = []
        for i in range(len(corr_cols)):
            for j in range(i+1, len(corr_cols)):
                r_corr = r_corr_matrix[i,j]
                s_corr = s_corr_matrix[i,j]
                diff = abs(r_corr - s_corr)
                pairs.append((corr_cols[i], corr_cols[j], r_corr, s_corr, diff))
                print(f"  {corr_cols[i]}-{corr_cols[j]}: {r_corr:.3f}->{s_corr:.3f} (Δ:{diff:.3f})")

        corr_fidelity = 1 - np.mean([p[4] for p in pairs])
        print(f"  Correlation Fidelity: {corr_fidelity*100:.1f}% ({len(pairs)} pairs)")
        corr_preservation = corr_fidelity
    else:
        print("  Insufficient numeric columns")
        corr_preservation = None
    
    # Privacy metrics
    print(f"\nPRIVACY METRICS")
    print(f"Real duplicates:  {df_real.duplicated().sum()}")
    print(f"Synth duplicates: {df_synth.duplicated().sum()}")
    
    num_rows = len(df_real_r)
    real_tuples = set(tuple(row) for row in df_real.itertuples(index=False, name=None))
    synth_tuples = set(tuple(row) for row in df_synth.itertuples(index=False, name=None))
    common_tuples = real_tuples.intersection(synth_tuples)
    identical = len(common_tuples)
    
    privacy_percent = (1 - identical/num_rows) * 100
    print(f"Identical rows: {identical}/{num_rows} | Privacy: {privacy_percent:4.1f}%")
    
    # Comprehensive summary
    print(f"\nCOMPREHENSIVE SUMMARY")
    print(f"  NMAE:  {nmae_mean:.3f} -> {(1-nmae_mean)*100:4.1f}%")
    print(f"  KS:    {ks_mean:.3f}")
    print(f"  TVD:   {tvd_mean:.3f} -> {(1-tvd_mean)*100:4.1f}%")
    if corr_preservation is not None:
        print(f"  Corr:  {corr_preservation*100:.1f}%")
    else:
        print(f"  Corr:  N/A")
    print(f"  Privacy:{privacy_percent:4.1f}%")
    
    # Final SDFS
    print("\n" + "="*90)
    sdfs = 1 - ((nmae_mean if not np.isnan(nmae_mean) else 0) * 0.4 +
                (tvd_mean if not np.isnan(tvd_mean) else 0) * 0.4 +
                (ks_mean if not np.isnan(ks_mean) else 0) * 0.2)
    
    sdfs_percent = sdfs * 100
    
    print("SYNTHETIC DATA FIDELITY SCORE (SDFS)")
    print(f"FINAL QUALITY: {sdfs_percent:5.1f}%")
    
    if sdfs > 0.90:
        print("EXCELLENT (>90%) - Production Ready")
    elif sdfs > 0.80:
        print("GOOD (80-90%) - Excellent for testing")
    elif sdfs > 0.70:
        print("ADEQUATE (70-80%) - Usable")
    else:
        print("POOR (<70%) - Needs improvement")
    
    print(f"XGBoostSynthesizer = {'PRODUCTION READY' if sdfs > 0.8 else 'OPTIMIZE'}")
    
    return {
        'sdfs': sdfs,
        'sdfs_percent': sdfs_percent,
        'nmae_mean': nmae_mean,
        'ks_mean': ks_mean,
        'tvd_mean': tvd_mean,
        'privacy_percent': privacy_percent,
        'corr_preservation': corr_preservation,
        'nmae_scores': nmae_scores,
        'ks_scores': ks_scores,
        'tvd_scores': tvd_scores
    }


# ==============================================
# SDV EVALUATION FUNCTION
# ==============================================

def sdv_comparison(df_real: pd.DataFrame, df_synth: pd.DataFrame):
    """
    Complete SDV evaluation for synthetic data quality
    """
    from sdv.metadata import SingleTableMetadata
    from sdv.evaluation.single_table import run_diagnostic, evaluate_quality

    print("SDV EVALUATION COMPLETE")
    print("=" * 80)

    # Metadata
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(df_real)
    print("Metadata OK")

    # Diagnostic Report
    print("\nDIAGNOSTIC REPORT")
    diagnostic = run_diagnostic(
        real_data=df_real,
        synthetic_data=df_synth,
        metadata=metadata
    )
    diagnostic_score = diagnostic.get_score()    
    print(f"DIAGNOSTIC SCORE: {diagnostic_score:.1%}")

    # Quality Report
    print("\nQUALITY REPORT")
    quality_report = evaluate_quality(
        real_data=df_real,
        synthetic_data=df_synth,
        metadata=metadata
    )
    quality_score = quality_report.get_score()

    # Verdict
    print("\n" + "=" * 80)
    print("SYNTHESIZER VERDICT")

    if quality_score >= 0.85:
        verdict = "EXCELLENT - Production Ready!"
    elif quality_score >= 0.70:
        verdict = "GOOD - Usable"
    else:
        verdict = "ADEQUATE - Needs optimization"

    print(verdict)
    print(f"Quality:    {quality_score:.1%}")
    print(f"Diagnostic: {diagnostic_score:.1%}")

    return {
        "quality_score": quality_score,
        "diagnostic_score": diagnostic_score,
        "quality_report": quality_report
    }


# ==============================================
# EXECUTE EVALUATIONS
# ==============================================

print("\nRunning Comprehensive Quality Comparison...")
results = comprehensive_comparison(df_real, df_synth)
print(f"\nFinal SDFS: {results['sdfs_percent']:.1f}%")

# Save quality report to file
report_text = f"""SYNTHETIC DATA QUALITY REPORT
Real data: {input_file_name}
Synthetic data: {output_file_name}
SDFS Score: {results['sdfs_percent']:.1f}%
NMAE: {results['nmae_mean']:.4f}
TVD: {results['tvd_mean']:.4f}
KS: {results['ks_mean']:.4f}
Privacy: {results['privacy_percent']:.1f}%
"""

report_filename = f"Quality_Report_M10_TH20_R{R}_PR{PR}_4CAT_MISS_X2_{synthesizer_type}_O{selected_order}.txt"
report_file_path = os.path.join(output_dir, report_filename)

with open(report_file_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"\nQuality report saved to: {report_file_path}")

# Run SDV evaluation if available
try:
    print("\nRunning SDV Evaluation...")
    sdv_results = sdv_comparison(df_real, df_synth)
    print(f"\nSDV Final Score: {sdv_results['quality_score']:.1%}")
    print(f"SDV Quality Report saved.")
except Exception as e:
    print(f"SDV evaluation not available: {e}")


# ==============================================
# DUPLICATE ANALYSIS
# ==============================================

print("\n" + "="*60)
print("DUPLICATE ANALYSIS")
print("="*60)

# Count occurrences of each record in synthetic data
count_df = (
    df_synth
    .groupby(list(df_synth.columns))
    .size()
    .reset_index(name='occurrence_count')
)

total_records = len(df_synth)
unique_records = (count_df['occurrence_count'] == 1).sum()

duplicates = count_df.loc[count_df['occurrence_count'] > 1, 'occurrence_count']
duplicate_rows_count = duplicates.count()
duplicate_sum = duplicates.sum()

print(f"Total records: {total_records}")
print(f"Unique records: {unique_records}")
print(f"Records with duplicates (rows): {duplicate_rows_count}")
print(f"Records with duplicates (total occurrences): {duplicate_sum}")

# Show duplicate records sorted by occurrence count
duplicate_df = (
    count_df
    .query('occurrence_count > 1')
    .sort_values('occurrence_count', ascending=False)
    .reset_index(drop=True)
)

if len(duplicate_df) > 0:
    print(f"\nDuplicate records found: {len(duplicate_df)}")
else:
    print("\nNo duplicate records found.")


# ==============================================
# ENTROPY ANALYSIS (OPTIONAL)
# ==============================================

def entropy_analysis(df: pd.DataFrame, target_col: str = 'diagnosis'):
    """
    Calculate entropy and Mutual Information for dataset features
    """
    from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
    from sklearn.preprocessing import LabelEncoder
    
    print("\n" + "="*60)
    print("ENTROPY AND MUTUAL INFORMATION ANALYSIS")
    print("="*60)
    
    def entropy_discrete(series):
        probs = series.dropna().value_counts(normalize=True)
        return -np.sum(probs * np.log2(probs)) if len(probs) > 0 else 0
    
    entropies = []
    max_entropies = []
    norm_entropies = []
    
    for col in df.columns:
        s = df[col].dropna()
        H = entropy_discrete(s)
        entropies.append(H)
        k = s.nunique()
        H_max = np.log2(k) if k > 1 else 0
        max_entropies.append(H_max)
        norm_entropies.append(H / H_max if H_max > 0 else 0)
    
    entropy_summary = pd.DataFrame({
        'Column': df.columns,
        'Data_Type': df.dtypes.values,
        'Total_Values': len(df),
        'Unique_Values': df.nunique().values,
        'Percent_Unique': (df.nunique() / len(df) * 100).round(2),
        'Null_Percent': (df.isnull().sum() / len(df) * 100).round(1),
        'Entropy': np.round(entropies, 3),
        'Max_Entropy': np.round(max_entropies, 3),
        'Norm_Entropy': np.round(norm_entropies, 3)
    })
    
    # Calculate Mutual Information
    X = df.drop(columns=[target_col]).copy()
    y = df[target_col]
    
    le_dict = {}
    for col in X.select_dtypes(include='object').columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        le_dict[col] = le
    
    mi = mutual_info_classif(X, y, discrete_features='auto', random_state=42)
    mi_df = pd.DataFrame({'Column': X.columns, 'MI': mi})
    
    entropy_summary = entropy_summary.merge(mi_df, on='Column', how='left')
    
    # Sort by MI descending
    entropy_summary = entropy_summary.sort_values('MI', ascending=False).reset_index(drop=True)
    
    print("\nFeature order (most informative to least):")
    print(entropy_summary['Column'].tolist())
    
    return entropy_summary

# Run entropy analysis
entropy_results = entropy_analysis(df_real)
display(entropy_results)

print("\n" + "="*60)
print("PROCESS COMPLETED SUCCESSFULLY")
print("="*60)

['age', 'physical_activity', 'genetic_predisposition', 'municipality_residence', 'civil_status', 'gender', 'occupation', 'diagnosis']
0
/home/onyxia/work/data/Step1/Output/real_data_datasetM10_TH20_RY_PR40_4CAT_MISS_X2.csv
/home/onyxia/work/data/Step2/Output/synthetic_data_datasetM10_TH20_RY_PR40_4CAT_MISS_X2_XGBoost_O0.csv
municipality_residence    object
age                       object
civil_status              object
gender                    object
occupation                object
physical_activity         object
genetic_predisposition    object
diagnosis                 object
dtype: object
Incremental synthesis order: ['age', 'physical_activity', 'genetic_predisposition', 'municipality_residence', 'civil_status', 'gender', 'occupation', 'diagnosis']
Synthesizing physical_activity
Categorical column detected
Synthesizing genetic_predisposition
Categorical column detected
Synthesizing municipality_residence
Categorical column detected
Synthesizing civil_status
Categorical column d

/opt/python/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/python/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



Feature order (most informative to least):
['physical_activity', 'genetic_predisposition', 'age', 'civil_status', 'municipality_residence', 'occupation', 'gender', 'diagnosis']


,Column,Data_Type,Total_Values,Unique_Values,Percent_Unique,Null_Percent,Entropy,Max_Entropy,Norm_Entropy,MI
0,physical_activity,object,10000,6,0.06,0.0,2.113,2.585,0.818,0.102866
1,genetic_predisposition,object,10000,9,0.09,0.0,2.756,3.170,0.869,0.091738
2,age,object,10000,48,0.48,0.0,5.554,5.585,0.995,0.014111
3,civil_status,object,10000,5,0.05,0.0,1.315,2.322,0.566,0.000000
4,municipality_residence,object,10000,145,1.45,0.0,6.317,7.180,0.880,0.000000
5,occupation,object,10000,2,0.02,0.0,0.938,1.000,0.938,0.000000
6,gender,object,10000,2,0.02,0.0,1.000,1.000,1.000,0.000000
7,diagnosis,object,10000,4,0.04,0.0,2.000,2.000,1.000,NaN



PROCESS COMPLETED SUCCESSFULLY
